GOLD SALES PERFORMACE

In [0]:
from pyspark.sql.functions import *

silver_sales = spark.read.table("workspace.default.silver_sales_new")

gold_sales_performance = (
    silver_sales
    .withColumn("sale_year", year(col("sale_date")))
    .withColumn("sale_month", month(col("sale_date")))
    .withColumn("sale_quarter",
        concat_ws("Q",
            year(col("sale_date")).cast("string"),
            quarter(col("sale_date")).cast("string")))
    .withColumn("sale_month_label",
        date_format(col("sale_date"), "yyyy-MM"))

    .groupBy(
        "sale_year","sale_month","sale_month_label","sale_quarter",
        "model_code","model_name","fuel_type",
        "dealer_region","channel","payment_mode","variant"
    )
    .agg(
        countDistinct("vin").alias("units_sold"),
        round(sum("sale_amount"), 2).alias("gross_revenue"),
        round(sum("net_revenue"), 2).alias("net_revenue"),
        round(sum("discount"), 2).alias("total_discount"),
        round(sum("tax_amount"), 2).alias("total_tax"),
        round(avg("sale_amount"), 2).alias("avg_sale_amount"),
        round(avg("discount"), 2).alias("avg_discount"),
        countDistinct("model_code").alias("models_sold"),
        countDistinct("dealer_id").alias("active_dealers"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .withColumn("discount_pct",
        round((col("total_discount")/col("gross_revenue"))*100,2))
    .withColumn("revenue_per_unit",
        round(col("net_revenue")/col("units_sold"),2))
    .withColumn("_processed_ts", current_timestamp())
)

gold_sales_performance.write.mode("overwrite").saveAsTable("workspace.default.gold_sales_performance")

GOLD PRODUCTION EFFICIENCY

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.gold_production_efficiency")

silver_prod = spark.table("workspace.default.silver_production_new")

gold_production_efficiency = (
    silver_prod
    .withColumn("prod_year", year(col("production_date")))
    .withColumn("prod_month", month(col("production_date")))
    .withColumn("prod_month_label",
        date_format(col("production_date"), "yyyy-MM"))
    .withColumn("prod_quarter",
        concat_ws("Q",
            year(col("production_date")).cast("string"),
            quarter(col("production_date")).cast("string")))

    .groupBy(
        "prod_year","prod_month","prod_month_label","prod_quarter",
        "plant_id","assembly_line","shift",
        "model_code","model_name","fuel_type","status"
    )
    .agg(
        count("production_id").alias("total_units"),
        sum("is_completed").alias("completed_units"),
        sum("is_delayed").alias("delayed_units"),
        round(avg("production_time_minutes"),2).alias("avg_production_time_minutes"),
        round(min("production_time_minutes"),2).alias("min_production_time"),
        round(max("production_time_minutes"),2).alias("max_production_time"),
        countDistinct("vin").alias("unique_vins_produced")
    )
    .withColumn("completion_rate_pct",
        round((col("completed_units")/col("total_units"))*100,2))
    .withColumn("delay_rate_pct",
        round((col("delayed_units")/col("total_units"))*100,2))
    .withColumn("_processed_ts", current_timestamp())
)

gold_production_efficiency.write.mode("overwrite").saveAsTable("workspace.default.gold_production_efficiency")


GOLD WARRANTY CLAIMS

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.gold_warranty_claims")

silver_warranty = spark.table("workspace.default.silver_warranty_new")
silver_service  = spark.table("workspace.default.silver_service_new")

svc_summary = (
    silver_service
    .groupBy("dealer_id")
    .agg(
        round(avg("customer_feedback_rating"),2).alias("avg_service_feedback"),
        count("service_id").alias("total_service_visits"),
        round(avg("service_cost"),2).alias("avg_service_cost")
    )
)

gold_warranty_claims = (
    silver_warranty
    .withColumn("claim_year", year(col("claim_date")))
    .withColumn("claim_month", month(col("claim_date")))
    .withColumn("claim_month_label",
        date_format(col("claim_date"), "yyyy-MM"))
    .withColumn("claim_quarter",
        concat_ws("Q",
            year(col("claim_date")).cast("string"),
            quarter(col("claim_date")).cast("string")))

    .groupBy(
        "claim_year","claim_month","claim_month_label","claim_quarter",
        "dealer_id","dealer_name","dealer_region",
        "model_code","model_name","fuel_type",
        "part_id","part_name","part_category","claim_status"
    )
    .agg(
        count("claim_id").alias("total_claims"),
        sum("is_approved").alias("approved_claims"),
        sum("is_rejected").alias("rejected_claims"),
        round(sum("claim_amount"),2).alias("total_claim_amount"),
        round(avg("claim_amount"),2).alias("avg_claim_amount"),
        round(max("claim_amount"),2).alias("max_claim_amount"),
        countDistinct("vin").alias("unique_vehicles_claimed")
    )
    .withColumn("approval_rate_pct",
        round((col("approved_claims")/col("total_claims"))*100,2))
    .withColumn("rejection_rate_pct",
        round((col("rejected_claims")/col("total_claims"))*100,2))
    .withColumn("pending_claims",
        col("total_claims") - col("approved_claims") - col("rejected_claims"))

    .join(svc_summary, "dealer_id", "left")
    .withColumn("_processed_ts", current_timestamp())
)

gold_warranty_claims.write.mode("overwrite").saveAsTable("workspace.default.gold_warranty_claims")

GOLD DEALER SCORECARD

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.gold_dealer_scorecard")

silver_sales = spark.table("workspace.default.silver_sales_new")
silver_service = spark.table("workspace.default.silver_service_new")
silver_warranty = spark.table("workspace.default.silver_warranty_new")
silver_inventory = spark.table("workspace.default.silver_inventory_new")

# Sales aggregation
sales_agg = (
    silver_sales
    .withColumn("sale_month_label", date_format(col("sale_date"), "yyyy-MM"))
    .withColumn("sale_year", year(col("sale_date")))
    .withColumn("sale_month", month(col("sale_date")))
    .groupBy("dealer_id","dealer_name","dealer_region",
             "dealer_type","dealer_rating",
             "sale_year","sale_month","sale_month_label")
    .agg(
        countDistinct("vin").alias("units_sold"),
        round(sum("net_revenue"),2).alias("total_revenue"),
        round(avg("sale_amount"),2).alias("avg_sale_amount"),
        round(avg("discount"),2).alias("avg_discount"),
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("model_code").alias("models_sold")
    )
)

# Service aggregation
service_agg = (
    silver_service
    .withColumn("sale_month_label", date_format(col("service_date"), "yyyy-MM"))
    .groupBy("dealer_id","sale_month_label")
    .agg(
        count("service_id").alias("service_visits"),
        round(avg("service_cost"),2).alias("avg_service_cost"),
        round(avg("customer_feedback_rating"),2).alias("avg_feedback_score"),
        sum("is_warranty_claim").alias("warranty_backed_services")
    )
)

# Warranty aggregation
warranty_agg = (
    silver_warranty
    .withColumn("sale_month_label", date_format(col("claim_date"), "yyyy-MM"))
    .groupBy("dealer_id","sale_month_label")
    .agg(
        count("claim_id").alias("warranty_claims_filed"),
        round(sum("claim_amount"),2).alias("total_warranty_cost"),
        round(avg("is_approved")*100,2).alias("warranty_approval_rate_pct")
    )
)

# Inventory
inventory_health = (
    silver_inventory
    .groupBy("dealer_id")
    .agg(
        sum("dealer_available_stock").alias("total_parts_stock"),
        round(avg("is_below_reorder")*100,2).alias("pct_parts_below_reorder"),
        round(sum("inventory_value"),2).alias("total_inventory_value")
    )
)

gold_dealer_scorecard = (
    sales_agg
    .join(service_agg, ["dealer_id","sale_month_label"], "left")
    .join(warranty_agg, ["dealer_id","sale_month_label"], "left")
    .join(inventory_health, "dealer_id", "left")

    .withColumn("composite_score",
        round(
            (when(col("units_sold") >= 50, 40)
             .otherwise((col("units_sold")/50)*40)) +
            (when(col("avg_feedback_score").isNotNull(),
                  (col("avg_feedback_score")/5)*30).otherwise(0)) +
            (when(col("warranty_approval_rate_pct").isNotNull(),
                  col("warranty_approval_rate_pct")/5).otherwise(0)) +
            (when(col("pct_parts_below_reorder").isNotNull(),
                  ((100-col("pct_parts_below_reorder"))/100)*10).otherwise(10))
        ,2)
    )

    .withColumn("performance_tier",
        when(col("composite_score") >= 80, "PLATINUM")
        .when(col("composite_score") >= 60, "GOLD")
        .when(col("composite_score") >= 40, "SILVER")
        .otherwise("BRONZE")
    )

    .withColumn("_processed_ts", current_timestamp())
)

gold_dealer_scorecard.write.mode("overwrite").saveAsTable("workspace.default.gold_dealer_scorecard")